# Calendar Dimension Table

Builds `dbw_caresync_dev.silver.calendar`, a Delta date dimension with one row per day from 1940-01-01 to 2030-12-31 (inclusive). Weeks follow ISO 8601 (Monday start); `day_of_week` is 1 = Monday ... 7 = Sunday.

In [0]:
-- One row per day from 1940-01-01 to 2030-12-31 (inclusive).
-- sequence() emits each date exactly once, so the table has no duplicate dates.
CREATE OR REPLACE TABLE dbw_caresync_dev.silver.calendar
USING DELTA
COMMENT 'Standard calendar date dimension, one row per date from 1940-01-01 to 2030-12-31. ISO 8601 weeks (Monday start).'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS date
)
SELECT
  CAST(date_format(date, 'yyyyMMdd') AS INT)                AS date_key,
  date,
  year(date)                                                 AS year,
  quarter(date)                                              AS quarter,
  concat('Q', quarter(date))                                 AS quarter_name,
  concat(year(date), '-Q', quarter(date))                    AS year_quarter,
  CASE WHEN month(date) <= 6 THEN 1 ELSE 2 END               AS half_year,
  month(date)                                                AS month,
  date_format(date, 'MMMM')                                  AS month_name,
  date_format(date, 'MMM')                                   AS month_short_name,
  date_format(date, 'yyyy-MM')                               AS year_month,
  CAST(date_format(date, 'yyyyMM') AS INT)                   AS year_month_key,
  weekofyear(date)                                           AS week,             -- ISO week number (1-53)
  EXTRACT(YEAROFWEEK FROM date)                              AS iso_year,         -- year the ISO week belongs to
  concat(EXTRACT(YEAROFWEEK FROM date), '-W', lpad(weekofyear(date), 2, '0')) AS iso_year_week,
  day(date)                                                  AS day_of_month,
  dayofyear(date)                                            AS day_of_year,
  weekday(date) + 1                                          AS day_of_week,      -- 1 = Monday ... 7 = Sunday
  date_format(date, 'EEEE')                                  AS day_name,
  date_format(date, 'EEE')                                   AS day_short_name,
  CAST(ceil(day(date) / 7.0) AS INT)                         AS week_of_month,
  date_sub(date, weekday(date))                              AS week_start_date,  -- Monday
  date_add(date_sub(date, weekday(date)), 6)                 AS week_end_date,    -- Sunday
  CAST(trunc(date, 'MM') AS DATE)                            AS month_start_date,
  last_day(date)                                             AS month_end_date,
  day(last_day(date))                                        AS days_in_month,
  CAST(trunc(date, 'QUARTER') AS DATE)                       AS quarter_start_date,
  date_sub(add_months(trunc(date, 'QUARTER'), 3), 1)         AS quarter_end_date,
  CAST(trunc(date, 'YEAR') AS DATE)                          AS year_start_date,
  make_date(year(date), 12, 31)                              AS year_end_date,
  weekday(date) >= 5                                         AS is_weekend,
  weekday(date) < 5                                          AS is_weekday,
  date = trunc(date, 'MM')                                   AS is_month_start,
  date = last_day(date)                                      AS is_month_end,
  date = trunc(date, 'QUARTER')                              AS is_quarter_start,
  date = date_sub(add_months(trunc(date, 'QUARTER'), 3), 1)  AS is_quarter_end,
  dayofyear(date) = 1                                        AS is_year_start,
  (month(date) = 12 AND day(date) = 31)                      AS is_year_end,
  (year(date) % 4 = 0 AND (year(date) % 100 <> 0 OR year(date) % 400 = 0)) AS is_leap_year
FROM dates;

In [0]:
-- Declare date_key as the (informational) primary key and make key columns non-nullable.
-- CTAS cannot declare constraints, so they are added here. DROP ... IF EXISTS keeps reruns from
-- failing if pk_calendar is still present (constraint names must be unique within the schema).
ALTER TABLE dbw_caresync_dev.silver.calendar DROP CONSTRAINT IF EXISTS pk_calendar;
ALTER TABLE dbw_caresync_dev.silver.calendar ALTER COLUMN date_key SET NOT NULL;
ALTER TABLE dbw_caresync_dev.silver.calendar ALTER COLUMN date SET NOT NULL;
ALTER TABLE dbw_caresync_dev.silver.calendar ADD CONSTRAINT pk_calendar PRIMARY KEY (date_key);

In [0]:
-- Expected: 33238 rows, 33238 distinct dates/date_keys, range 1940-01-01 to 2030-12-31
SELECT
  count(*)                AS row_count,
  count(DISTINCT date)    AS distinct_dates,
  count(DISTINCT date_key) AS distinct_date_keys,
  min(date)               AS min_date,
  max(date)               AS max_date,
  datediff(max(date), min(date)) + 1 AS expected_days
FROM dbw_caresync_dev.silver.calendar;

In [0]:
SELECT * FROM dbw_caresync_dev.silver.calendar ORDER BY date LIMIT 5;